# Verificación de los documentos de un postulante (Chile)

Compara cada documento que entregó el postulante (`postulant_files/<Nombre>/`: PDF nativo, PDF escaneado o fotos) con su dato oficial en JSON (`webscrap_files/<Nombre>/`) y emite un dictamen por documento y la aptitud para el cargo.

**Antes de empezar**
- Abra el notebook desde la carpeta raíz del proyecto, con un kernel que tenga instalado `requirements.txt`.
- Configure la celda de configuración y ejecute todas las celdas en orden (*Run All*).
- Los PDF nativos se leen directo (sin OCR); las fotos y escaneos pasan por el OCR en la GPU (máx. 2,5 GB de VRAM).


## 1. Configuración

- `POSTULANTE`: nombre de la carpeta del postulante dentro de `DIR_POSTULANTES`.
- `DIR_POSTULANTES` / `DIR_OFICIALES`: carpetas con los documentos entregados y con los JSON oficiales (una subcarpeta por postulante, con el mismo nombre).
- `HOY`: fecha de referencia para calcular vigencias.
- `USAR_GPU`: `False` para leer con CPU (más lento).


In [ ]:
from datetime import date
from pathlib import Path

RAIZ = Path.cwd()
POSTULANTE = "NOMBRE APELLIDO APELLIDO"
DIR_POSTULANTES = RAIZ / "postulant_files"
DIR_OFICIALES = RAIZ / "webscrap_files"
HOY = date.today()
USAR_GPU = True

## 2. Verificación

El RUT declarado y el cargo se toman del registro de postulantes (`postulant_files/candidatos.xlsx`) si el postulante está registrado.


In [ ]:
import sys
sys.path.insert(0, str(RAIZ))
from verificacion import config as C
from verificacion.chile import pipeline_cl as PL

registro = PL.leer_registro(C.REGISTRO_POSTULANTES)
personas, documentos, no_admitidos, meta = PL.verificar([POSTULANTE], registro, DIR_POSTULANTES, DIR_OFICIALES,
                                                        workers=4, usar_gpu=USAR_GPU, hoy=HOY)
p = personas[0]
print(f"{p['carpeta']} | RUT {p['dni'] or '(no registrado)'} | cargo: {p['puesto']}")
print(f"Autenticidad: {p['veredicto']} | Aptitud: {p['aptitud']}")
for m in p['motivos_aptitud']:
    print('  -', m)

## 3. Dictamen por documento


In [ ]:
for d in documentos:
    print(f"\n{d['archivo']}\n  {d['tipo_documento']} | {d['formato']} | {d['veredicto']} | dato oficial: {d['archivo_oficial'] or '-'}")
    for m in d['motivos']:
        print('   ·', m)
for x in no_admitidos:
    print(f"\nNO ADMITIDO: {x['archivo']} ({x['motivo']})")

## 4. Campo por campo

Solo los campos que no coinciden exactamente con el dato oficial.


In [ ]:
import pandas as pd
filas = [{"archivo": d['archivo'][:40], "campo": c['zona'], "oficial (JSON)": c['oficial'], "leído": c['postulante'],
          "estado": c['estado'], "motivo": c['motivo'], "confianza": c['conf']}
         for d in documentos for c in d['campos'] if c['estado'] not in ('IGUAL', 'NO_APLICA')]
pd.DataFrame(filas)